# Load packages and libraries

In [1]:
.libPaths()
assign(".lib.loc", "/home/manuel.tardaguila/conda_envs/multiome_NEW_downstream_analysis/lib/R/library", envir = environment(.libPaths))
.libPaths()
# sessionInfo()

Sys.setenv(RETICULATE_PYTHON="/home/manuel.tardaguila/conda_envs/multiome_NEW_downstream_analysis/bin/python")
library(reticulate)
reticulate::use_python("/home/manuel.tardaguila/conda_envs/multiome_NEW_downstream_analysis/bin/python")
reticulate::use_condaenv("/home/manuel.tardaguila/conda_envs/multiome_NEW_downstream_analysis")
reticulate::py_module_available(module='leidenalg')
reticulate::import('leidenalg') 
suppressMessages(library(hdf5r))
suppressMessages(library(Seurat)) 
suppressMessages(library(Signac)) 
suppressMessages(library(EnsDb.Hsapiens.v86)) 
suppressMessages(library(dplyr)) 
suppressMessages(library(ggplot2)) 
suppressMessages(library(Matrix)) 
suppressMessages(library(data.table)) 
suppressMessages(library(ggpubr)) 
suppressMessages(library(ggplot2))
suppressMessages(library(chromVAR))
suppressMessages(library(enrichR))
suppressMessages(library(JASPAR2020))
suppressMessages(library(TFBSTools))
suppressMessages(library(motifmatchr))
suppressMessages(library(BSgenome.Hsapiens.UCSC.hg38))
library(pheatmap)

[1] "/group/soranzo/conda_envs/multiome_NEW_downstream_analysis/lib/R/library"

[1] "/home/manuel.tardaguila/conda_envs/multiome_NEW_downstream_analysis/lib/R/library"

[1] TRUE

Module(leidenalg)

# Read in data

In [2]:
setwd("/group/soranzo/manuel.tardaguila/2025_K562_multiome_reanalysis/Downstream_analysis/")

In [3]:
adata<-readRDS(file="merged_clusters_final_annotated.rds")

In [4]:
adata

An object of class Seurat 
459097 features across 11250 samples within 4 assays 
Active assay: SCT (29123 features, 3000 variable features)
 3 layers present: counts, data, scale.data
 3 other assays present: RNA, RNA_raw, ATAC
 5 dimensional reductions calculated: pca, umap.rna, lsi, umap.atac, umap.wnn

# Find motifs in peaks

In [5]:
DefaultAssay(adata) <- 'ATAC'

In [6]:
pfm <- getMatrixSet(
  x = JASPAR2020,
  opts = list(collection = "CORE", tax_group = 'vertebrates', all_versions = FALSE))

In [7]:
#str(pfm)

In [8]:
motifdata = cbind(sapply(pfm, function(x) unlist(x@name)),sapply(pfm, function(x) unlist(x@matrixClass )))

In [9]:
#str(motifdata)

In [10]:
TF_motifs<-unlist(motifdata)

TF_motifs[grep("CUX1|RUNX1",TF_motifs)]

[1] "RUNX1" "CUX1"

In [11]:
names(pfm) <- motifdata[,1]

# Add to adata

In [12]:
adata <- AddMotifs(
  object = adata, genome =BSgenome.Hsapiens.UCSC.hg38 , assay= "ATAC",
  pfm = pfm
)

Building motif matrix

Finding motif positions

Creating Motif object



In [13]:
#str(adata)

# chromVar step

In [14]:
#### long step
DefaultAssay(adata) <- 'ATAC'
adata <- RunChromVAR(
  object = adata,
  genome = BSgenome.Hsapiens.UCSC.hg38
)

Computing GC bias per region

Selecting background regions

Computing deviations from background

Constructing chromVAR assay

Warning message:
"Layer counts isn't present in the assay object; returning NULL"


In [15]:
DefaultAssay(adata) <- 'chromvar'

In [16]:
devscores = GetAssayData(adata,layer='data', assay="chromvar")

In [17]:
str(devscores)

 num [1:746, 1:11250] -1.1901 -0.7441 0.0309 -1.2753 -3.6719 ...
 - attr(*, "dimnames")=List of 2
  ..$ : chr [1:746] "Arnt" "Ahr::Arnt" "Ddit3::Cebpa" "Mecom" ...
  ..$ : chr [1:11250] "MCO_1278_AAACAGCCAAGGTCCT-1" "MCO_1278_AAACAGCCATGGTTAT-1" "MCO_1278_AAACATGCAGAAATGC-1" "MCO_1278_AAACCAACACATAACT-1" ...


In [18]:
which(row.names(devscores) == 'CUX1')

which(row.names(devscores) == 'RUNX1')

[1] 243

[1] 41

In [19]:
#devscores[243,]

# Save data

In [20]:
output_dir<-"/group/soranzo/manuel.tardaguila/2025_K562_multiome_reanalysis/Downstream_analysis/chromvar_analysis/"

In [21]:
write.table(devscores, file=file.path(output_dir,'chromVAR_dev_scores_j2020.txt'), sep='\t', quote=FALSE)


In [35]:
setwd("/group/soranzo/manuel.tardaguila/2025_K562_multiome_reanalysis/Downstream_analysis/")

In [36]:
saveRDS(adata, file="merged_clusters_final_annotated_motifs_and_chromvar.rds")

# Pick-up after calculation of scores

In [4]:
setwd("/group/soranzo/manuel.tardaguila/2025_K562_multiome_reanalysis/Downstream_analysis/")

adata<-readRDS(file="merged_clusters_final_annotated_motifs_and_chromvar.rds")

adata

Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


Loading required package: Signac

Loading required package: Seurat



An object of class Seurat 
459843 features across 11250 samples within 5 assays 
Active assay: chromvar (746 features, 0 variable features)
 1 layer present: data
 4 other assays present: RNA, RNA_raw, ATAC, SCT
 5 dimensional reductions calculated: pca, umap.rna, lsi, umap.atac, umap.wnn

# chromvar analysis

## Subset to cluster 1

In [5]:
adata_sub_cluster_1<-subset(adata, seurat_clusters == 1)

adata_sub_cluster_1

Idents(adata_sub_cluster_1)<- "Genotype"


summary(adata_sub_cluster_1@meta.data$Genotype)


An object of class Seurat 
459843 features across 2811 samples within 5 assays 
Active assay: chromvar (746 features, 0 variable features)
 1 layer present: data
 4 other assays present: RNA, RNA_raw, ATAC, SCT
 5 dimensional reductions calculated: pca, umap.rna, lsi, umap.atac, umap.wnn

wt rs139141690_HET     rs139141690        Del_16bp        Del_80bp 
            834             494             396             172             915

### wt vs Del_16bp

In [6]:
# Set the chromvar assay as the default for this analysis
DefaultAssay(adata_sub_cluster_1) <- "chromvar"

# Find differentially active motifs between two cell types (e.g., "T cells" vs "B cells")
# This is analogous to FindMarkers for gene expression, but now on motif deviation scores
diff_motifs <- FindMarkers(
  object = adata_sub_cluster_1,
  ident.1 = "wt",
  ident.2 = "Del_16bp",
  group.by = "Genotype", # Or whatever metadata column defines your groups
  test.use = "wilcox",    # Or "bimod", "t", etc.
  min.pct = 0.1,          # Minimum percentage of cells in either group expressing the feature
  logfc.threshold = 0.25  # Minimum log2 fold-change for motif deviation
)

diff_motifs$motif<-row.names(diff_motifs)
str(diff_motifs)
indexes<-grep("CUX1|RUNX1",diff_motifs$motif)
diff_motifs[indexes,]

Warning message:
“The `slot` argument of `GetAssayData()` is deprecated as of SeuratObject 5.0.0.
ℹ Please use the `layer` argument instead.
ℹ The deprecated feature was likely used in the Seurat package.
  Please report the issue at <https://github.com/satijalab/seurat/issues>.”
Warning message:
“`PackageCheck()` was deprecated in SeuratObject 5.0.0.
ℹ Please use `rlang::check_installed()` instead.
ℹ The deprecated feature was likely used in the Seurat package.
  Please report the issue at <https://github.com/satijalab/seurat/issues>.”
For a (much!) faster implementation of the Wilcoxon Rank Sum Test,
(default method for FindMarkers) please install the presto package
--------------------------------------------
install.packages('devtools')
devtools::install_github('immunogenomics/presto')
--------------------------------------------
After installation of presto, Seurat will automatically use the more 
efficient implementation (no further action necessary).
This message will be shown o

'data.frame':	567 obs. of  6 variables:
 $ p_val     : num  3.10e-08 6.08e-08 1.14e-07 1.27e-07 2.01e-07 ...
 $ avg_log2FC: num  6.42 3.36 6.16 1.65 -1.56 ...
 $ pct.1     : num  0.673 0.288 0.665 0.297 0.348 0.637 0.721 0.428 0.67 0.629 ...
 $ pct.2     : num  0.471 0.541 0.477 0.5 0.541 0.436 0.506 0.645 0.494 0.442 ...
 $ p_val_adj : num  2.31e-05 4.54e-05 8.49e-05 9.47e-05 1.50e-04 ...
 $ motif     : chr  "GATA4" "GABPA" "GATA2" "ELF1" ...


,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,motif
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
RUNX1,0.023815,6.081845,0.366,0.453,1,RUNX1


### wt vs Del_80bp

In [7]:
# Set the chromvar assay as the default for this analysis
DefaultAssay(adata_sub_cluster_1) <- "chromvar"

# Find differentially active motifs between two cell types (e.g., "T cells" vs "B cells")
# This is analogous to FindMarkers for gene expression, but now on motif deviation scores
diff_motifs <- FindMarkers(
  object = adata_sub_cluster_1,
  ident.1 = "wt",
  ident.2 = "Del_80bp",
  group.by = "Genotype", # Or whatever metadata column defines your groups
  test.use = "t",    # Or "bimod", "t", etc.
  min.pct = 0.1,          # Minimum percentage of cells in either group expressing the feature
  logfc.threshold = 0.25  # Minimum log2 fold-change for motif deviation
)

diff_motifs$motif<-row.names(diff_motifs)
str(diff_motifs)

indexes<-grep("CUX1|RUNX1",diff_motifs$motif)
diff_motifs[indexes,]

Warning message in mean.fxn(object[features, cells.2, drop = FALSE]):
“NaNs produced”


'data.frame':	552 obs. of  6 variables:
 $ p_val     : num  2.23e-23 2.51e-23 4.68e-22 5.84e-22 9.24e-22 ...
 $ avg_log2FC: num  -2.17 -4.24 -2.63 -2.34 -3.83 ...
 $ pct.1     : num  0.288 0.376 0.375 0.297 0.348 0.393 0.394 0.385 0.397 0.387 ...
 $ pct.2     : num  0.525 0.564 0.538 0.499 0.525 0.552 0.553 0.554 0.55 0.55 ...
 $ p_val_adj : num  1.66e-20 1.87e-20 3.49e-19 4.35e-19 6.89e-19 ...
 $ motif     : chr  "GABPA" "ERF" "ETV5" "ELF1" ...


p_val,avg_log2FC,pct.1,pct.2,p_val_adj,motif
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>


### wt vs rs139141690

In [8]:
# Set the chromvar assay as the default for this analysis
DefaultAssay(adata_sub_cluster_1) <- "chromvar"

# Find differentially active motifs between two cell types (e.g., "T cells" vs "B cells")
# This is analogous to FindMarkers for gene expression, but now on motif deviation scores
diff_motifs <- FindMarkers(
  object = adata_sub_cluster_1,
  ident.1 = "wt",
  ident.2 = "rs139141690",
  group.by = "Genotype", # Or whatever metadata column defines your groups
  test.use = "t",    # Or "bimod", "t", etc.
  min.pct = 0.1,          # Minimum percentage of cells in either group expressing the feature
  logfc.threshold = 0.25  # Minimum log2 fold-change for motif deviation
)

diff_motifs$motif<-row.names(diff_motifs)
str(diff_motifs)
cat("\n")
indexes<-grep("CUX1|RUNX1",diff_motifs$motif)
diff_motifs[indexes,]

Warning message in mean.fxn(object[features, cells.2, drop = FALSE]):
“NaNs produced”


'data.frame':	519 obs. of  6 variables:
 $ p_val     : num  4.03e-18 2.67e-16 3.01e-16 5.26e-16 4.20e-15 ...
 $ avg_log2FC: num  -4.12 -2.77 -3.22 -4.76 -2.59 ...
 $ pct.1     : num  0.348 0.375 0.412 0.376 0.398 0.394 0.385 0.403 0.302 0.354 ...
 $ pct.2     : num  0.535 0.571 0.614 0.576 0.616 0.578 0.588 0.591 0.503 0.533 ...
 $ p_val_adj : num  3.01e-15 1.99e-13 2.24e-13 3.93e-13 3.13e-12 ...
 $ motif     : chr  "ETV6" "ETV5" "ETS2" "ERF" ...



,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,motif
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
RUNX1,1.019454e-08,-3.2784138,0.366,0.530,7.605128e-06,RUNX1
CUX1,1.914905e-04,-0.4950702,0.537,0.619,1.428519e-01,CUX1


## Subset to cluster 3

In [9]:
adata_sub_cluster_3<-subset(adata, seurat_clusters == 3)

adata_sub_cluster_3

Idents(adata_sub_cluster_3)<- "Genotype"


summary(adata_sub_cluster_3@meta.data$Genotype)


An object of class Seurat 
459843 features across 1584 samples within 5 assays 
Active assay: chromvar (746 features, 0 variable features)
 1 layer present: data
 4 other assays present: RNA, RNA_raw, ATAC, SCT
 5 dimensional reductions calculated: pca, umap.rna, lsi, umap.atac, umap.wnn

wt rs139141690_HET     rs139141690        Del_16bp        Del_80bp 
            454             132             379             111             508

### wt vs Del_16bp

In [10]:
# Set the chromvar assay as the default for this analysis
DefaultAssay(adata_sub_cluster_3) <- "chromvar"

# Find differentially active motifs between two cell types (e.g., "T cells" vs "B cells")
# This is analogous to FindMarkers for gene expression, but now on motif deviation scores
diff_motifs <- FindMarkers(
  object = adata_sub_cluster_3,
  ident.1 = "wt",
  ident.2 = "Del_16bp",
  group.by = "Genotype", # Or whatever metadata column defines your groups
  test.use = "t",    # Or "bimod", "t", etc.
  min.pct = 0.1,          # Minimum percentage of cells in either group expressing the feature
  logfc.threshold = 0.25  # Minimum log2 fold-change for motif deviation
)

diff_motifs$motif<-row.names(diff_motifs)
str(diff_motifs)
indexes<-grep("CUX1|RUNX1",diff_motifs$motif)
diff_motifs[indexes,]

Warning message in mean.fxn(object[features, cells.1, drop = FALSE]):
“NaNs produced”
Warning message in mean.fxn(object[features, cells.2, drop = FALSE]):
“NaNs produced”


'data.frame':	606 obs. of  6 variables:
 $ p_val     : num  4.21e-09 5.24e-09 8.11e-09 1.82e-08 3.18e-08 ...
 $ avg_log2FC: num  -2.66 -2.16 -1.92 -2.22 -4.91 ...
 $ pct.1     : num  0.381 0.39 0.396 0.385 0.399 0.381 0.39 0.319 0.482 0.361 ...
 $ pct.2     : num  0.649 0.685 0.676 0.667 0.685 0.631 0.649 0.568 0.703 0.586 ...
 $ p_val_adj : num  3.14e-06 3.91e-06 6.05e-06 1.36e-05 2.37e-05 ...
 $ motif     : chr  "ERF" "ELK4" "ETS1" "ZBTB7A" ...


,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,motif
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
RUNX1,1.244185e-06,-2.6703993,0.588,0.757,0.0009281622,RUNX1
CUX1,3.080949e-01,-0.2777076,0.630,0.559,1.0000000000,CUX1


### wt vs Del_80bp

In [11]:
# Set the chromvar assay as the default for this analysis
DefaultAssay(adata_sub_cluster_3) <- "chromvar"

# Find differentially active motifs between two cell types (e.g., "T cells" vs "B cells")
# This is analogous to FindMarkers for gene expression, but now on motif deviation scores
diff_motifs <- FindMarkers(
  object = adata_sub_cluster_3,
  ident.1 = "wt",
  ident.2 = "Del_80bp",
  group.by = "Genotype", # Or whatever metadata column defines your groups
  test.use = "t",    # Or "bimod", "t", etc.
  min.pct = 0.1,          # Minimum percentage of cells in either group expressing the feature
  logfc.threshold = 0.25  # Minimum log2 fold-change for motif deviation
)

diff_motifs$motif<-row.names(diff_motifs)
str(diff_motifs)
indexes<-grep("CUX1|RUNX1",diff_motifs$motif)
diff_motifs[indexes,]

Warning message in mean.fxn(object[features, cells.1, drop = FALSE]):
“NaNs produced”
Warning message in mean.fxn(object[features, cells.2, drop = FALSE]):
“NaNs produced”


'data.frame':	591 obs. of  6 variables:
 $ p_val     : num  3.25e-32 2.30e-31 9.40e-31 1.63e-30 4.71e-30 ...
 $ avg_log2FC: num  -2.91 -2.7 -3.19 -4.45 -1.64 ...
 $ pct.1     : num  0.381 0.399 0.291 0.319 0.39 0.392 0.396 0.381 0.385 0.308 ...
 $ pct.2     : num  0.683 0.687 0.654 0.657 0.689 0.697 0.663 0.679 0.673 0.64 ...
 $ p_val_adj : num  2.43e-29 1.71e-28 7.01e-28 1.22e-27 3.51e-27 ...
 $ motif     : chr  "ERF" "ETV5" "ETV1" "GABPA" ...


,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,motif
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
RUNX1,2.184512e-16,-3.0837722,0.588,0.783,1.629646e-13,RUNX1
CUX1,7.259871e-01,-0.4038293,0.630,0.640,1.000000e+00,CUX1


### wt vs rs139141690

In [12]:
# Set the chromvar assay as the default for this analysis
DefaultAssay(adata_sub_cluster_3) <- "chromvar"

# Find differentially active motifs between two cell types (e.g., "T cells" vs "B cells")
# This is analogous to FindMarkers for gene expression, but now on motif deviation scores
diff_motifs <- FindMarkers(
  object = adata_sub_cluster_3,
  ident.1 = "wt",
  ident.2 = "rs139141690",
  group.by = "Genotype", # Or whatever metadata column defines your groups
  test.use = "t",    # Or "bimod", "t", etc.
  min.pct = 0.1,          # Minimum percentage of cells in either group expressing the feature
  logfc.threshold = 0.25  # Minimum log2 fold-change for motif deviation
)

diff_motifs$motif<-row.names(diff_motifs)
str(diff_motifs)
indexes<-grep("CUX1|RUNX1",diff_motifs$motif)
diff_motifs[indexes,]

Warning message in mean.fxn(object[features, cells.1, drop = FALSE]):
“NaNs produced”
Warning message in mean.fxn(object[features, cells.2, drop = FALSE]):
“NaNs produced”


'data.frame':	496 obs. of  6 variables:
 $ p_val     : num  5.82e-23 7.30e-22 7.92e-21 8.85e-21 8.76e-20 ...
 $ avg_log2FC: num  -2.2 -1.84 -2.92 -4.64 -1.45 ...
 $ pct.1     : num  0.392 0.352 0.319 0.392 0.399 0.291 0.515 0.308 0.454 0.39 ...
 $ pct.2     : num  0.652 0.615 0.586 0.67 0.67 0.562 0.739 0.554 0.675 0.665 ...
 $ p_val_adj : num  4.34e-20 5.44e-19 5.91e-18 6.61e-18 6.53e-17 ...
 $ motif     : chr  "ETV6" "ELF3" "GABPA" "EHF" ...


,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,motif
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
RUNX1,7.008065e-07,-0.4642716,0.588,0.744,0.0005228016,RUNX1
CUX1,1.498794e-02,-0.8323959,0.630,0.697,1.0000000000,CUX1
